<a href="https://colab.research.google.com/github/uomna/flyrank-ml/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/uomna/flyrank-ml/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

Looking at the distributions of key fields before testing any signal.
avg_position and ctr both show heavy right/left tails typical of search
data: most content sits at weak positions with low CTR, while a small
number of top-ranking pages pull the averages. impressions_90d is
extremely right-skewed — a handful of high-traffic pages carry a huge
share of total impressions, so any score using raw impressions needs a
threshold or log transform to avoid being dominated by a few outliers.

In [5]:
!git clone https://github.com/uomna/flyrank-ml.git repo
%cd repo
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df_valid = df[df["avg_position"] > 0].copy()

print("avg_position distribution:")
print(df_valid["avg_position"].describe())

print("\nctr distribution:")
print(df_valid["ctr"].describe())

print("\nimpressions_90d distribution (note the heavy right tail):")
print(df_valid["impressions_90d"].describe())
print("Top 1% threshold:", df_valid["impressions_90d"].quantile(0.99))
print("Median:", df_valid["impressions_90d"].median())

Cloning into 'repo'...
remote: Enumerating objects: 203, done.
remote: Counting objects: 100% (203/203), done.
remote: Compressing objects: 100% (159/159), done.
remote: Total 203 (delta 91), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (203/203), 1.97 MiB | 5.49 MiB/s, done.
Resolving deltas: 100% (91/91), done.
/content/repo/repo
avg_position distribution:
count    28795.000000
mean        17.026268
std         15.152439
min          0.100000
25%          6.700000
50%         11.400000
75%         22.900000
max        245.000000
Name: avg_position, dtype: float64

ctr distribution:
count    28795.000000
mean         0.519662
std          3.232606
min          0.000000
25%          0.000000
50%          0.080000
75%          0.300000
max        100.000000
Name: ctr, dtype: float64

impressions_90d distribution (note the heavy right tail):
count     28795.000000
mean       5417.909984
std       17152.423172
min           1.000000
25%         118.000000
50%      

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

Signal 1 — Freshness vs CTR: MIXED
Freshness tier does not show a consistent direction with CTR. The 0-30d
tier has moderate CTR (0.61%), but 31-90d and 91-180d tiers are lower
(0.12% and 0.24%). The 181+ tier shows the highest CTR (3.69%), but this
group is very small (n=174) compared to the 0-30d tier (n=20,480) —
likely survivorship bias (only strong, well-performing old pages remain
live), not evidence that staleness improves performance.

Signal 2 — Position vs CTR: CONFIRMED
CTR decreases consistently and monotonically as position gets worse.
Top 3 positions average 2.71% CTR (n=1,141), dropping to 0.65% at
positions 4-10 (n=11,842), 0.32% at 11-20 (n=7,273), and 0.21% at 21+
(n=8,539). This is a clean, monotonic relationship with no reversal or
small-sample noise — the clearest signal in this dataset.

Signal 3 — Word count vs engagement rate: MIXED
Longer content does not consistently show higher engagement. Content
under 1,500 words and content over 4,000 words show similar engagement
rates, with the middle range (2,000-3,000 words, where most content
actually sits) showing no clear advantage either way. Word length alone
is not a reliable signal for engagement in this dataset — consistent
with the Methodology decision (ML-08) not to weight word_count heavily
in the model.

In [6]:
# Signal 1 — Freshness vs CTR
signal1 = df_valid.groupby("freshness_tier").agg(
    n=("content_id", "count"), avg_ctr=("ctr", "mean")
).reset_index()
print("Signal 1 — Freshness vs CTR:")
print(signal1)

# Signal 2 — Position vs CTR
df_valid["position_bucket"] = pd.cut(
    df_valid["avg_position"], bins=[0, 3, 10, 20, float('inf')],
    labels=["1-3", "4-10", "11-20", "21+"]
)
signal2 = df_valid.groupby("position_bucket", observed=True).agg(
    n=("content_id", "count"), avg_ctr=("ctr", "mean")
).reset_index()
print("\nSignal 2 — Position vs CTR:")
print(signal2)

# Signal 3 — Word count vs engagement rate
df_valid["word_count_bucket"] = pd.cut(
    df_valid["word_count"], bins=[0, 1500, 2000, 3000, 4000, float('inf')],
    labels=["<1500", "1500-2000", "2000-3000", "3000-4000", "4000+"]
)
signal3 = df_valid.groupby("word_count_bucket", observed=True).agg(
    n=("content_id", "count"), avg_engagement=("engagement_rate", "mean")
).reset_index()
print("\nSignal 3 — Word count vs Engagement rate:")
print(signal3)

Signal 1 — Freshness vs CTR:
  freshness_tier      n   avg_ctr
0           0-30  19300  0.629417
1           181+    158  3.856329
2          31-90    175  0.117543
3         91-180   9162  0.238601

Signal 2 — Position vs CTR:
  position_bucket      n   avg_ctr
0             1-3   1141  2.714303
1            4-10  11842  0.651045
2           11-20   7273  0.323443
3             21+   8539  0.211333

Signal 3 — Word count vs Engagement rate:
  word_count_bucket     n  avg_engagement
0             <1500  2704        2.956938
1         1500-2000  1408        2.818317
2         2000-3000  7911        3.326076
3         3000-4000  5003        2.324475
4             4000+  4083        1.511166


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

FlyRank's real flag (used in this capstone's baseline, ML-07):
ctr_below_position_tier — a page is flagged if its CTR falls below the
average CTR for its position tier, assuming that a page ranking well but
getting fewer clicks than peers at the same rank has a fixable
title/snippet problem.

Does the data support this assumption? Partially. Signal 2 above confirms
the underlying premise — CTR does fall predictably as position tier
worsens, so "expected CTR by tier" is a meaningful baseline to compare
against. But the flag's implicit assumption that a below-tier-average
page is fixable via title/meta is not directly testable with this data:
two of the ten highest-scored pages under this exact rule (see ML-07 top
10) had near-zero CTR (0.01-0.03%) despite good position, which looks
more like a possible tracking issue than a title problem. The rule
correctly identifies underperformance relative to position, but cannot
by itself distinguish a content/title problem from a measurement problem
— that distinction needs human review, which is why the action playbook
(ML-10) treats every flagged row as a hypothesis, not a verdict.

In [7]:
# Recompute the flag exactly as in ML-07, and check how many flagged
# pages have suspiciously near-zero CTR (a measurement-problem signal)
df_valid["tier_avg_ctr"] = df_valid.groupby("position_bucket", observed=True)["ctr"].transform("mean")
df_valid["ctr_gap"] = df_valid["tier_avg_ctr"] - df_valid["ctr"]

visible = (df_valid["impressions_90d"] >= 500).astype(int)
underperforming = (df_valid["ctr_gap"] > 0).astype(int)
df_valid["flag_score"] = underperforming * visible * df_valid["ctr_gap"] * df_valid["impressions_90d"]

flagged = df_valid[df_valid["flag_score"] > 0].sort_values("flag_score", ascending=False)
top10 = flagged.head(10)

near_zero_ctr = (top10["ctr"] < 0.05).sum()
print(f"Total pages flagged by the rule: {len(flagged)}")
print(f"Of the top 10 flagged pages, {near_zero_ctr} have CTR under 0.05% "
      f"(possible tracking issue, not a title problem)")
print("\nTop 10 flagged pages, position and CTR:")
print(top10[["content_id", "avg_position", "ctr", "tier_avg_ctr"]].to_string(index=False))

Total pages flagged by the rule: 13678
Of the top 10 flagged pages, 2 have CTR under 0.05% (possible tracking issue, not a title problem)

Top 10 flagged pages, position and CTR:
          content_id  avg_position  ctr  tier_avg_ctr
content_8c19996aa890           2.5 0.15      2.714303
content_4c36c775b818           2.3 0.41      2.714303
content_8451fc6f034d           2.3 0.03      2.714303
content_44e481c8f55b           1.4 0.65      2.714303
content_9532f197bbc8           2.0 0.87      2.714303
content_e12868d1f396           2.9 0.07      2.714303
content_4a6607efcb46           2.2 0.01      2.714303
content_4fc39a2b8cf0           2.6 0.69      2.714303
content_11900bd7941a           2.8 0.41      2.714303
content_03d2673b2553           1.9 0.83      2.714303


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

For a content team, the practical takeaway is: trust position-based CTR
gaps as a starting filter (Signal 2 is the strongest, cleanest pattern in
this data), but do not trust content age or word count alone as decision
signals (Signals 1 and 3 are too mixed to act on directly). Before acting
on any CTR-gap flag, a reviewer should spot-check the raw CTR number —
values under roughly 0.05% at a decent position are more likely a
tracking or indexing issue than a title problem, and deserve a Search
Console check before a rewrite is scheduled.

In [8]:
# Quantify the practical recommendation: how many flagged pages fall
# into the "possible tracking issue" zone across the full flagged set
suspect_rate = (flagged["ctr"] < 0.05).mean()
suspect_count = (flagged["ctr"] < 0.05).sum()

print(f"Of {len(flagged)} pages flagged by the CTR-gap rule, "
      f"{suspect_count} ({suspect_rate:.1%}) have CTR under 0.05%.")
print("These are candidates for a tracking-verification step before "
      "being routed to a title/meta rewrite.")

Of 13678 pages flagged by the CTR-gap rule, 3413 (25.0%) have CTR under 0.05%.
These are candidates for a tracking-verification step before being routed to a title/meta rewrite.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.